# 2_models/04 — Full-cohort held-out risk scores

Runs `pipelines.training.run_full_cohort_risk_scores` for every `(scheme, event)` pair whose
full-cohort CV training has completed, producing per-patient held-out risk scores for the text and
base models so the two can be compared on the same cohort (e.g. stratified KM curves).

**Runs after** the full-cohort training arrays and **before** `3_biomarkers/01` (the biomarker pipeline consumes
these scores).

An event is eligible only once `text_val.csv` exists — those CV results are what the script reads to
pick the best hyperparameters, so a pair without them has nothing to score from. Completed pairs
(both `text_risk_scores.csv` and `base_risk_scores.csv` present) are skipped unless `OVERWRITE`.

Pending events run **concurrently**, one single-core subprocess each — see the run cell for how the
pool is sized. Tasks are independent (each writes only its own `(scheme, event)` output directory),
so concurrency costs nothing in correctness.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<22} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def report_outputs(outputs: list[tuple[str, str]]) -> None:
    """Print size and mtime for each (label, path) that exists."""
    for label, path in outputs:
        if os.path.exists(path):
            mb = os.path.getsize(path) / 1e6
            mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(path)))
            print(f"[ok     ] {label:<26} {mb:>8.1f} MB   {mtime}")
        else:
            print(f"[missing] {label:<26} {path}")


def run_module(module: str, args: list[str] | None = None, env: dict | None = None,
               capture: bool = False) -> dict:
    """Run `python -m module` from REPO_ROOT. Returns {returncode, wall_s, stdout}."""
    cmd = [sys.executable, "-m", module, *(args or [])]
    started = time.perf_counter()
    run_env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}
    kwargs = dict(cwd=str(REPO_ROOT), env=run_env)
    if capture:
        kwargs.update(text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    proc = subprocess.run(cmd, **kwargs)
    return {"returncode": proc.returncode, "wall_s": time.perf_counter() - started,
            "stdout": (proc.stdout or "") if capture else ""}


print(f"repo root: {REPO_ROOT}")
print(f"Python:  {sys.executable}")

## Configuration

In [ ]:
from schemes import SCHEMES as SCHEME_CONFIG, full_cohort_risk_dir, scheme_results_dir

MODULE = "pipelines.training.run_full_cohort_risk_scores"

SCHEMES = sorted(SCHEME_CONFIG.keys())   # subset to restrict the run
ANCHOR = "treatment"                     # see anchors.py: "treatment" or "sequencing"

OVERWRITE = False
MAX_ITER = 1000
BACKEND = "threading"
VERBOSE = False     # True replays each subprocess's captured output when it finishes

# Two axes of parallelism are available and they compete for the same cores:
#   N_JOBS      — CV folds within one task (what the SLURM array uses, at 6 CPUs/task)
#   MAX_WORKERS — tasks in flight at once (what this notebook uses)
# With hundreds of independent tasks, running many single-core tasks beats running a
# few wide ones, so N_JOBS is pinned to 1 here. Raising both oversubscribes the node.
N_JOBS = 1

# None = auto: leave a core free, capped at ~8G/worker (array_full_cohort_risk_scores.sh
# reserves --mem=8G per task; each task loads the full embedding prediction dataset).
MAX_WORKERS = None

print(f"module:  {MODULE}")
print(f"schemes: {', '.join(SCHEMES)}")
print(f"anchor:  {ANCHOR}")
print(f"n_jobs:  {N_JOBS}   workers: {MAX_WORKERS or 'auto'}")

## Preconditions

The training output roots the eligible pairs are discovered from. A missing root means that scheme's
full-cohort arrays have not run at all — not that the risk-score step failed. Does not raise.

In [ ]:
# scheme_results_dir is read-only; get_output_dir would makedirs and report every
# scheme as present.
check_inputs([(f"training root ({scheme})",
               os.path.join(scheme_results_dir(scheme, ANCHOR), "full_cohort"))
              for scheme in SCHEMES])

## Discover eligible (scheme, event) pairs

In [ ]:
def trained_events(scheme: str) -> list[str]:
    """Events whose full-cohort CV wrote text_val.csv — the input this step reads."""
    train_root = os.path.join(scheme_results_dir(scheme, ANCHOR), "full_cohort")
    if not os.path.isdir(train_root):
        return []
    return sorted(ev for ev in os.listdir(train_root)
                  if os.path.exists(os.path.join(train_root, ev, "text_val.csv")))


def risk_done(scheme: str, event: str) -> bool:
    risk_dir = full_cohort_risk_dir(scheme, event, ANCHOR)
    return all(os.path.exists(os.path.join(risk_dir, f))
               for f in ("text_risk_scores.csv", "base_risk_scores.csv"))


tasks: list[tuple[str, str]] = []
for scheme in SCHEMES:
    events = trained_events(scheme)
    pending = [ev for ev in events if OVERWRITE or not risk_done(scheme, ev)]
    print(f"{scheme:<14} {len(events):>4} trained, {len(pending):>4} pending "
          f"({len(events) - len(pending)} already scored)")
    tasks.extend((scheme, ev) for ev in pending)

print(f"\nQueue: {len(tasks)} task(s)")

## Run

One subprocess per event, run concurrently across `MAX_WORKERS` processes, so a failure on one does
not kill the queue. Each task is pinned to a single core (`N_JOBS = 1`, and BLAS/Polars thread
counts pinned in the environment), so throughput tracks worker count.

Pinning `POLARS_MAX_THREADS`/`RAYON_NUM_THREADS` is not optional under concurrency: Polars sizes its
Rayon pool to the machine's core count at import, which with several tasks in flight is hundreds of
threads and trips "can't start new thread" before any fitting begins. The SLURM script does not
pin them because there one task owns the node.

`MAX_WORKERS = None` auto-sizes to one less than the CPU count, capped at ~8G/worker. On a shared
node, size it to what you actually reserved, not what `os.cpu_count()` reports.

Output is captured either way — it is the only record of why a task failed, and the summary below
reads it — but echoed only for failures, or for everything when `VERBOSE`. Under concurrency, live
streaming would interleave unreadably.

In [ ]:
import concurrent.futures

from tqdm.auto import tqdm

FAIL_TAIL_LINES = 15

args_base = ["--anchor", ANCHOR, "--max-iter", str(MAX_ITER), "--backend", BACKEND,
             "--n-jobs", str(N_JOBS)]
if OVERWRITE:
    args_base.append("--overwrite")

# Each task is single-core by construction; without this, every subprocess would size its
# BLAS and Rayon pools to the whole machine and they would fight for the same cores.
SINGLE_THREAD_ENV = {var: "1" for var in (
    "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS",
    "POLARS_MAX_THREADS", "RAYON_NUM_THREADS")}


def resolve_workers(n_tasks: int) -> int:
    """Auto-size the pool: leave a core free, and budget ~8G/task (the SLURM array's --mem)."""
    if MAX_WORKERS is not None:
        return max(1, min(int(MAX_WORKERS), n_tasks))
    workers = max(1, (os.cpu_count() or 1) - 1)
    try:
        # Linux-only; on a cgroup-limited node this reflects the real allowance better
        # than total system memory. Skipped silently where unavailable.
        available_gb = (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_AVPHYS_PAGES")) / 1024 ** 3
        workers = max(1, min(workers, int(available_gb // 8)))
    except (ValueError, OSError, AttributeError):
        pass
    return max(1, min(workers, n_tasks))


def run_task(scheme: str, event: str) -> dict:
    outcome = run_module(MODULE, ["--scheme", scheme, "--event", event, *args_base],
                         env=SINGLE_THREAD_ENV, capture=True)
    outcome.update(scheme=scheme, event=event)
    return outcome


n_workers = resolve_workers(len(tasks))
print(f"Running {len(tasks)} task(s) across {n_workers} worker(s)"
      + (" (OVERWRITE=True)" if OVERWRITE else ""))

run_started = time.perf_counter()
results, n_failed = [], 0

if tasks:
    # Threads only wait on subprocess.run -- the work is in separate processes, so the GIL
    # is irrelevant and a thread pool avoids pickling overhead.
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as pool:
        in_flight = {pool.submit(run_task, *task): task for task in tasks}
        with tqdm(total=len(tasks), desc="events", unit="event") as bar:
            try:
                for future in concurrent.futures.as_completed(in_flight):
                    result = future.result()
                    results.append(result)
                    label = f"{result['scheme']}:{result['event']}"
                    if result["returncode"] != 0:
                        n_failed += 1
                        bar.write(f"[fail] {label} (exit {result['returncode']})")
                    if VERBOSE:
                        bar.write(f"\n{'=' * 72}\n{label} (exit {result['returncode']}, "
                                  f"{result['wall_s'] / 60:.1f}m)\n{'=' * 72}")
                        bar.write(result["stdout"].rstrip())
                    bar.update(1)
                    bar.set_postfix_str(f"{n_failed} failed" if n_failed else "", refresh=True)
            except KeyboardInterrupt:
                # Without this, pool shutdown would block on every queued task.
                for future in in_flight:
                    future.cancel()
                print("\nInterrupted -- cancelled queued tasks, waiting for in-flight ones.")
                raise

run_elapsed = time.perf_counter() - run_started
failed = [r for r in results if r["returncode"] != 0]
print(f"\nRan {len(results)} task(s) in {run_elapsed / 60:.1f} min across {n_workers} worker(s) — "
      f"{len(results) - len(failed)} succeeded, {len(failed)} failed")

for result in failed:
    lines = result["stdout"].splitlines()
    tail = lines[-FAIL_TAIL_LINES:]
    print(f"\n{'-' * 72}\n{result['scheme']}:{result['event']} (exit {result['returncode']}) — "
          f"last {len(tail)} of {len(lines)} output line(s)\n{'-' * 72}")
    print("\n".join(tail) if tail else "(no output captured)")

## Summary

Coverage per scheme on disk now, then the hand-off check. A scheme short of full coverage leaves `3_biomarkers/01`
working from a partial set of risk scores.

In [ ]:
total_done = total_trained = 0
for scheme in SCHEMES:
    events = trained_events(scheme)
    n_done = sum(1 for ev in events if risk_done(scheme, ev))
    total_done += n_done
    total_trained += len(events)
    pct = 100.0 * n_done / len(events) if events else 0.0
    print(f"{scheme:<14} {n_done:>4}/{len(events):<4} events scored  ({pct:>5.1f}%)")

print(f"\nOverall: {total_done}/{total_trained} events have risk scores")

if failed:
    print(f"\n{len(failed)} failure(s) this run:")
    for result in failed:
        print(f"  {result['scheme']}:{result['event']} (exit {result['returncode']})")

print("\n" + ("Ready for 3_biomarkers/01." if total_done else
              "Not ready — no risk scores on disk; 3_biomarkers/01 has nothing to consume."))